In [1]:
import os
import pandas as pd
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.http import models
from qdrant_client.models import Distance, VectorParams
import uuid

# ===========================
# تنظیمات مسیرها
# ===========================
QDRANT_URL = "http://localhost:6333"
EMBEDDINGS_DIR = r"F:\Thesis\project\2-RAG\vector_store\vector_store_bge"
LAWS_ROOT_DIR = r"F:\Thesis\project\2-RAG\raw_laws"

# تعریف ۳ کالکشن
COLLECTION_LAWS = "legal_laws"             # قوانین
COLLECTION_UNITY = "legal_votes_unity"     # وحدت رویه (دیوان عالی + عدالت)
COLLECTION_DADNAMEH = "legal_votes_dadnameh"  # دادنامه‌ها

VECTOR_SIZE = 1024

# نگاشت فایل‌های خاص به کالکشن‌های مربوطه
SPECIAL_FILES_CONFIG = {
    "dadnameh_metadata": {
        "path": r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\dadnameh_metadata.tsv",
        "collection": COLLECTION_DADNAMEH
    },
    "vahdat_divan-ali_metadata": {
        "path": r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\vahdat_divan-ali_metadata.tsv",
        "collection": COLLECTION_UNITY
    },
    "vahdat_edalat-edari_metadata": {
        "path": r"F:\Thesis\project\2-RAG\raw_laws\Unifying Precedent Decisions\preprocessed_data\vahdat_edalat-edari_metadata.tsv",
        "collection": COLLECTION_UNITY
    }
}

# ===========================
# توابع
# ===========================
def find_law_tsv(npy_filename, root_dir):
    base_name = npy_filename.replace("_embeddings.npy", "")
    for root, dirs, files in os.walk(root_dir):
        if f"{base_name}.tsv" in files:
            return os.path.join(root, f"{base_name}.tsv")
    return None

def upload_to_qdrant(client, collection_name, df, embeddings, text_col, source_name):
    if len(df) != len(embeddings):
        print(f"!! Error mismatch: {source_name}")
        return

    points = []
    for idx, row in df.iterrows():
        vec = embeddings[idx].tolist()
        payload = row.to_dict()
        payload["page_content"] = str(row.get(text_col, ""))
        payload["source_file"] = source_name

        clean_payload = {k: v for k, v in payload.items() if pd.notna(v)}

        point_id = str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{source_name}_{idx}"))

        points.append(
            models.PointStruct(id=point_id, vector=vec, payload=clean_payload)
        )

    for i in range(0, len(points), 100):
        client.upsert(collection_name=collection_name, points=points[i : i + 100])
    print(f"   -> Uploaded {len(points)} to [{collection_name}]")

# ===========================
# اجرا روی Qdrant Docker
# ===========================
client = QdrantClient(url=QDRANT_URL)

# 1. ساخت ۳ کالکشن (اگر وجود دارند، پاک و دوباره ساخته می‌شوند)
all_cols = [COLLECTION_LAWS, COLLECTION_UNITY, COLLECTION_DADNAMEH]
for col in all_cols:
    if client.collection_exists(col):
        client.delete_collection(col)
    client.create_collection(
        collection_name=col,
        vectors_config=VectorParams(size=VECTOR_SIZE, distance=Distance.COSINE),
    )
    print(f"Created Collection: {col}")

# 2. پردازش فایل‌های NPY و TSV
npy_files = [f for f in os.listdir(EMBEDDINGS_DIR) if f.endswith(".npy")]

for npy_file in npy_files:
    base_name = npy_file.replace("_embeddings.npy", "")
    npy_path = os.path.join(EMBEDDINGS_DIR, npy_file)
    embeddings = np.load(npy_path)

    print(f"\nProcessing: {base_name}")

    if base_name in SPECIAL_FILES_CONFIG:
        config = SPECIAL_FILES_CONFIG[base_name]
        target_collection = config["collection"]
        tsv_path = config["path"]

        print(f" -> Type: VOTE/UNITY -> Collection: {target_collection}")
        if os.path.exists(tsv_path):
            df = pd.read_csv(tsv_path, sep="\t")
            upload_to_qdrant(client, target_collection, df, embeddings, "vote_text", base_name)
        else:
            print("!! TSV missing")
    else:
        tsv_path = find_law_tsv(npy_file, LAWS_ROOT_DIR)
        if tsv_path:
            print(f" -> Type: LAW -> Collection: {COLLECTION_LAWS}")
            df = pd.read_csv(tsv_path, sep="\t")
            if "text" in df.columns:
                upload_to_qdrant(client, COLLECTION_LAWS, df, embeddings, "text", base_name)
        else:
            print(" -> Skipped (No TSV found)")

print("\nDone! 3 Collections Created on Docker Qdrant.")


Created Collection: legal_laws
Created Collection: legal_votes_unity
Created Collection: legal_votes_dadnameh

Processing: aeename-bimeh_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 101 to [legal_laws]

Processing: aeename-hemayat-family-93_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 69 to [legal_laws]

Processing: aeename-kanun-vokala-1400_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 164 to [legal_laws]

Processing: aeename-mafad-asnad-rasmi_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 204 to [legal_laws]

Processing: aeename-mojer-mostajer-76_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 19 to [legal_laws]

Processing: aeename-pishfroosh-sakhteman_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 22 to [legal_laws]

Processing: aeename-sabt-amlak_articles
 -> Type: LAW -> Collection: legal_laws
   -> Uploaded 162 to [legal_laws]

Processing: aeename-tashkil-dadgah_articles
 ->

In [2]:
from qdrant_client import QdrantClient
q = QdrantClient(url="http://localhost:6333")
print("\n📊 Collections:")
print(q.get_collections())


📊 Collections:
collections=[CollectionDescription(name='legal_votes_dadnameh'), CollectionDescription(name='legal_votes_unity'), CollectionDescription(name='legal_laws')]


In [4]:
import os
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient


# ===========================
# تنظیمات
# ===========================
QDRANT_URL = "http://localhost:6333"

COLLECTION_LAWS = "legal_laws"
COLLECTION_UNITY = "legal_votes_unity"
COLLECTION_DADNAMEH = "legal_votes_dadnameh"

MODEL_NAME = "baai/bge-m3"

OPENROUTER_EMBEDDINGS_API_KEY = os.environ.get("OPENROUTER_EMBEDDINGS_API_KEY")
if not OPENROUTER_EMBEDDINGS_API_KEY:
    raise ValueError("OPENROUTER_EMBEDDINGS_API_KEY در محیط تنظیم نشده است.")

client_embed = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_EMBEDDINGS_API_KEY,
)

# اتصال به Qdrant روی Docker
qdrant = QdrantClient(url=QDRANT_URL)


# ===========================
# تابع گرفتن امبدینگ سوال
# ===========================
def embed_query(text: str) -> np.ndarray:
    resp = client_embed.embeddings.create(
        model=MODEL_NAME,
        input=[text]
    )
    emb = np.array(resp.data[0].embedding, dtype="float32")
    return emb


# ===========================
# تابع جستجو در یک کالکشن
# ===========================
def search_collection(collection_name: str, query_vector: np.ndarray, top_k: int = 5):
    result = qdrant.query_points(
        collection_name=collection_name,
        query=query_vector.tolist(),
        limit=top_k,
        with_payload=True,
    )
    return result.points


# ===========================
# تابع تست کامل
# ===========================
def test_query(query_text: str, top_k: int = 5):
    print("=" * 80)
    print("سؤال:")
    print(query_text.strip())
    print("=" * 80)

    q_emb = embed_query(query_text)

    for col_name, label in [
        (COLLECTION_LAWS, "قوانین"),
        (COLLECTION_UNITY, "آراء وحدت رویه"),
        (COLLECTION_DADNAMEH, "دادنامه‌ها"),
    ]:
        print(f"\n### جستجو در کالکشن: {label} ({col_name})")
        try:
            hits = search_collection(col_name, q_emb, top_k=top_k)
            if not hits:
                print("  - نتیجه‌ای پیدا نشد.")
                continue

            for rank, hit in enumerate(hits, start=1):
                payload = hit.payload or {}
                text = payload.get("page_content", "")[:200].replace("\n", " ")
                law_name = payload.get("law_name", "")
                article = payload.get("article_number", "") or payload.get("ruling_number", "") or ""

                print(f"  رتبه {rank} | امتیاز: {hit.score:.4f}")
                if law_name:
                    print(f"    منبع: {law_name} | شماره: {article}")
                print(f"    متن: {text}")
        except Exception as e:
            print(f"  خطا در جستجوی کالکشن {col_name}: {e}")


# ===========================
# اجرای چند تست نمونه
# ===========================

q1 = """
«منع تبذیر» ذیل ضوابط حاکم بر کدام یک از شئون نظام جمهوری اسلامی در قانون اساسی مورد اشاره قرار گرفته است؟
1) سیاست خارجی 
2) اقتصاد 
3) امنیت ملی 
4) فرهنگ
"""

q2 = """
در قانون اساسی کدام آزادی مشروط به رعایت حقوق دیگران شده است؟ 
1) آزادی عقیده 
2) آزادی اجتماعات 
3) آزادی بیان 
4) آزادی انتخاب شغل 
"""

q3 = """
بنا به تصریح قانون اساس ی مصوبات کدام مرجع نباید با متن و روح قانون مغایرت داشته باشد؟ 
1) شورای عالی امنیت ملی 
2) شوراهای فرعی شورای عالی امنیت ملی 
3) شوراهای محلی 
4) هیئت وزیران
"""

for q in [q1, q2, q3]:
    test_query(q, top_k=5)


سؤال:
«منع تبذیر» ذیل ضوابط حاکم بر کدام یک از شئون نظام جمهوری اسلامی در قانون اساسی مورد اشاره قرار گرفته است؟
1) سیاست خارجی 
2) اقتصاد 
3) امنیت ملی 
4) فرهنگ

### جستجو در کالکشن: قوانین (legal_laws)
  رتبه 1 | امتیاز: 0.6189
    منبع: قانون اساسی جمهوری اسلامی ایران | شماره: 
    متن: اصل چهارم کلیه قوانین و مقررات مدنی، جزایی، مالی، اقتصادی، اداری، فرهنگی، نظامی، سیاسی و غیر اینها باید بر اساس موازین اسلامی باشد. این اصل بر اطلاق یا عموم همه اصول قانون اساسی و قوانین و مقررات دیگر
  رتبه 2 | امتیاز: 0.5792
    منبع: قانون اساسی جمهوری اسلامی ایران | شماره: 
    متن: اصل چهل و سوم برای تأمین استقلال اقتصادی جامعه و ریشه کن کردن فقر و محرومیت و برآوردن نیازهای انسان در جریان رشد، با حفظ آزادی او، اقتصاد جمهوری اسلامی ایران بر اساس ضوابط زیر استوار می شود: 1 - تأمین
  رتبه 3 | امتیاز: 0.5718
    منبع: قانون آیین دادرسی کیفری | شماره: 4
    متن: ماده 4- اصل، برائت است. هرگونه اقدام محدودکننده، سالب آزادی و ورود به حریم خصوصی اشخاص جز به حکم قانون و با رعایت مقررات و تحت نظارت مقام ق